# Bölüm 7: Verinizi Hazırlama
**İlk LLM'inizi Oluşturun — Bölüm 7: Verinizi Hazırlama**

Bu defter veri hazırlama adımlarını gösterir: temizleme, tekilleştirme, bölme, parçalama ve hızlı istatistiklerle JSONL'ye kaydetme.
Küçük oyuncak veri satır içi; harici dosya gerekmez.

In [ ]:
# ===== İÇE AKTARMALAR =====
import re           # Metinde kalıp eşleştirme için düzenli ifadeler
import json         # JSON ve JSONL dosyalarını oku/yaz
import hashlib      # Kopya tespiti için parmak izleri oluştur
import unicodedata  # Unicode normalizasyonu (özel karakterleri işle)
from collections import Counter  # Kelime sıklıklarını say
import random       # Train/val/test bölmeleri için veriyi karıştır

print('Kurulum tamamlandı')

## Python Araçları Hızlı Referans

Bu defter birkaç Python aracı kullanır. İşte hızlı bir rehber:

**Düzenli İfadeler (regex):** Metinde kalıp tabanlı bul/değiştir
- `re.sub(pattern, replacement, text)` — `pattern`'in tüm eşleşmelerini bul ve değiştir
- Kalıp `<[^>]+>` şu anlama gelir: `<` bul, sonra `>` olmayan tüm karakterleri, sonra `>` → HTML etiketleriyle eşleşir

**Hashing:** Herhangi bir metin için benzersiz bir "parmak izi" oluştur
- Aynı metin → aynı hash (her zaman). Farklı metin → farklı hash (neredeyse her zaman)
- Tüm belgeleri karşılaştırmadan kopyaları tespit etmek için yararlı
- `hashlib.sha1(text.encode()).hexdigest()` → 40 karakterlik parmak izi

**JSON/JSONL:** Yapılandırılmış veriyi saklamak için veri formatları
- **JSON:** Tüm veriyle tek bir büyük dosya (tüm dosyayı belleğe yüklemek gerekir)
- **JSONL:** Satır başına bir JSON kaydı (satır satır akış yapılabilir = bellek verimli)

**Kümeler (Sets):** Kopyası olmayan koleksiyonlar, hızlı "X bu kümede mi?" kontrolü
- `seen = set()` sonra `seen.add(item)` ve `item in seen`

**Tür ipuçları** (`: str`, `: float = 0.8`): İnsanlar için dokümantasyon (Python bunları görmezden gelir)
- `text: str` "text bir string olmalı" demektir
- `train_p: float = 0.8` "train_p bir ondalıklı sayı olmalı, varsayılan 0.8" demektir

## Örnek metinler
Küçük bir derlem simüle etmek için birkaç oyuncak paragraf.

In [ ]:
raw_docs = [
    "THE TIME MACHINE — CHAPTER I\n\nThis   is   a    sample text… with   odd spacing, smart "quotes", and tabs\t.",
    "AI systems learn from examples. Data quality shapes model quality.",
    "The key to machine learning is data; the secret to building AI is understanding.",
]
print('Belgeler:', len(raw_docs))

## Temizleme & normalizasyon

HTML'i sil, kontrol karakterlerini kaldır, boşlukları daralt, tırnakları normalize et.

In [ ]:
def clean_text(text: str) -> str:
    """LLM eğitimi için metni temizle ve normalize et."""
    # Unicode normalizasyonu: ﬁ → fi, ｆｕｌｌ → full, vb.
    # NFKC = Uyumluluk ayrıştırması + Kanonik birleştirme
    text = unicodedata.normalize("NFKC", text)
    
    # HTML etiketlerini sil: <p>, <div>, <span class="foo">, vb.
    # Kalıp: < ve ardından > olmayan tüm karakterler, sonra >
    text = re.sub(r"<[^>]+>", " ", text)
    
    # Yeni satırları ve sekmeleri boşluklarla değiştir
    text = re.sub(r"[\n\t]", " ", text)
    
    # Birden fazla boşluğu bir boşluğa daralt, başındaki/sonundaki boşlukları kaldır
    text = re.sub(r"\s+", " ", text).strip()
    
    # Akıllı tırnakları düz tırnaklara normalize et
    text = text.replace("“", '"').replace("”", '"')
    
    return text

cleaned_docs = [clean_text(d) for d in raw_docs]
for i, d in enumerate(cleaned_docs):
    print(f"Temizlenmiş {i}: {d[:80]}...")

## Tekilleştirme

Paragrafları hash'le, tekrarları at.

In [ ]:
# SHA1 hash kullanarak metin için bir "parmak izi" oluştur
# Aynı metin → aynı parmak izi (her zaman)
# Farklı metin → farklı parmak izi (ezici olasılıkla)
def hash_chunk(text: str) -> str:
    # .encode("utf-8") string'i bayt'lara çevirir (hashlib tarafından gereklidir)
    # .hexdigest() hash'i 40 karakterlik string olarak döndürür
    return hashlib.sha1(text.encode("utf-8")).hexdigest()

# Hash'lemeyi göster
sample = "The cat sat on the mat."
print(f"Metin: '{sample}'")
print(f"Hash: {hash_chunk(sample)}")
print(f"Aynı metin aynı hash: {hash_chunk(sample) == hash_chunk(sample)}")
print(f"Farklı metin farklı hash: {hash_chunk(sample) != hash_chunk('Dog.')}")

def dedup_chunks(chunks):
    """Hash tabanlı parmak izi kullanarak kopya parçaları kaldır."""
    seen = set()    # Gördüğümüz hash'leri izle (hızlı arama!)
    unique = []     # Sadece benzersiz parçaları tut
    
    for c in chunks:
        h = hash_chunk(c)
        if h in seen:
            continue        # Kopyayı atla
        seen.add(h)         # Bu hash'i hatırla
        unique.append(c)    # Parçayı tut
    
    return unique

# Tekilleştirmenin çalıştığını kanıtlamak için bir kopya ekle
test_docs = cleaned_docs + [cleaned_docs[0]]  # İlk belgenin kopyasını ekle
deduped = dedup_chunks(test_docs)
print(f'\nTekilleştirme öncesi: {len(test_docs)} belge')
print(f'Tekilleştirme sonrası:  {len(deduped)} belge')

## Train/val/test bölmesi (belgeye göre)

İlgili metni bir arada tut; sızıntıyı önle.

In [ ]:
def split_docs(docs: list, train_p: float = 0.8, val_p: float = 0.1, seed: int = 42):
    """Belgeleri train/val/test kümelerine böl.
    
    Argümanlar:
        docs: Bölünecek belgelerin listesi
        train_p: Eğitim için oran (varsayılan 0.8 = %80)
        val_p: Doğrulama için oran (varsayılan 0.1 = %10)
        seed: Tekrarlanabilirlik için rastgele tohum
    
    Döndürür:
        train, val, test listeleri (test kalan oranı alır)
    """
    # Çağıranın listesini değiştirmemek için bir kopya üzerinde çalış
    docs = list(docs)
    random.seed(seed)
    random.shuffle(docs)
    
    n = len(docs)
    n_train = int(n * train_p)
    n_val = int(n * val_p)
    
    train = docs[:n_train]
    val = docs[n_train:n_train + n_val]
    test = docs[n_train + n_val:]
    
    return train, val, test

# Tekilleştirilmiş belgelerimize bölmeyi uygula
train_docs, val_docs, test_docs = split_docs(deduped, train_p=0.6, val_p=0.2, seed=42)
print(f'Eğitim: {len(train_docs)}, Doğrulama: {len(val_docs)}, Test: {len(test_docs)}')

## Bağlam pencereleri için parçalama
Parça sınırları boyunca bağlamı korumak için örtüşme ile uzun metni böl.

In [ ]:
def chunk_text(text: str, max_chars: int = 200, overlap: int = 50):
    """Metni örtüşen parçalara böl.
    
    Argümanlar:
        text: Parçalanacak girdi metni
        max_chars: Parça başına maksimum karakter
        overlap: Parçalar arasında örtüşecek karakter sayısı
    
    Döndürür:
        Metin parçalarının listesi
    """
    chunks = []
    start = 0
    while start < len(text):
        end = min(len(text), start + max_chars)
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        start += max_chars - overlap  # (max_chars - overlap) kadar ilerle
    return chunks

# Tüm bölmelere parçalama uygula
chunked = []
for split, docs in [('train', train_docs), ('val', val_docs), ('test', test_docs)]:
    for d in docs:
        for c in chunk_text(d, max_chars=120, overlap=30):
            chunked.append({'text': c, 'split': split, 'source': 'toy'})
            
print('Toplam parçalar:', len(chunked))

## Örtüşmeyi Görselleştir
Sınırlar boyunca bağlamı korumak için parçaların nasıl örtüştüğünü gör.

In [ ]:
# Net konumlara sahip bir test metni oluştur
test_text = "A" * 500  # 500 karakter
chunks = chunk_text(test_text, max_chars=200, overlap=50)

print(f"Metin uzunluğu: {len(test_text)}")
print(f"Parça sayısı: {len(chunks)}")
print(f"Parça uzunlukları: {[len(c) for c in chunks]}")

# Ardışık parçalar arasındaki örtüşmeyi doğrula
# Python dilim gösterimi:
#   text[-50:]  = son 50 karakter (negatif indeks sondan sayar)
#   text[:50]   = ilk 50 karakter
if len(chunks) >= 2:
    # Parça 0'ın son 50 karakteri parça 1'in ilk 50 karakterine eşit olmalı
    chunk0_end = chunks[0][-50:]    # Parça 0'ın son 50 karakteri
    chunk1_start = chunks[1][:50]   # Parça 1'in ilk 50 karakteri
    overlap_matches = chunk0_end == chunk1_start
    
    print(f"\nÖrtüşme doğrulaması: {overlap_matches}")
    print(f"Parça 0 şununla biter: ...{chunks[0][-10:]}")
    print(f"Parça 1 şununla başlar: {chunks[1][:10]}...")
    
# Gerçek metin ile
real_text = "This is sentence one. This is sentence two. This is sentence three." * 5
real_chunks = chunk_text(real_text, max_chars=100, overlap=30)
print(f"\nGerçek metin {len(real_chunks)} parçaya bölündü")
print(f"Parça 0: ...{real_chunks[0][-40:]}")
print(f"Parça 1: {real_chunks[1][:40]}...")
print("\n✅ Örtüşme parça sınırları boyunca bağlamı korur!")

## Kalite Kontrolleri & Akıl Sağlığı Doğrulaması
Boş parçaları, HTML sızıntısını ve boyut sorunlarını işaretleyen otomatik kontrollerle problemleri erken yakala.

In [ ]:
def sanity_check(chunks, stage_name):
    """Herhangi bir ardışık düzen aşamasında veri üzerinde akıl sağlığı kontrolleri çalıştır."""
    print(f"\n{'='*50}")
    print(f"Akıl Sağlığı Kontrolü: {stage_name}")
    print(f"{'='*50}")
    
    if not chunks:
        print("⚠️  UYARI: Parça yok!")
        return
    
    # Temel istatistikler
    print(f"✓ Toplam parçalar: {len(chunks)}")
    lengths = [len(c) if isinstance(c, str) else len(c.get('text', '')) for c in chunks]
    avg_len = sum(lengths) / len(lengths)
    print(f"✓ Ortalama uzunluk: {avg_len:.0f}")
    print(f"✓ Maksimum uzunluk: {max(lengths)}")
    print(f"✓ Minimum uzunluk: {min(lengths)}")
    
    # Sorunları kontrol et
    if max(lengths) > 10 * avg_len:
        print("⚠️  UYARI: Maksimum uzunluk ortalamanın 10 katı - parçalama bozuk olabilir")
    
    # HTML sızıntısını kontrol et
    texts = [c if isinstance(c, str) else c.get('text', '') for c in chunks]
    all_text = ' '.join(texts).lower()
    html_words = {'div', 'span', 'href', 'html', 'class', 'src'}
    found_html = [w for w in html_words if w in all_text]
    if found_html:
        print(f"⚠️  UYARI: HTML etiketleri bulundu: {found_html}")
    else:
        print("✓ HTML sızıntısı tespit edilmedi")
    
    # Boş parçaları kontrol et
    empty = sum(1 for l in lengths if l < 10)
    if empty > 0:
        print(f"⚠️  UYARI: {empty} parça < 10 karakter")
    else:
        print("✓ Boş parça yok")
    
    # Örnek
    sample = texts[0] if texts else "N/A"
    print(f"\n✓ Örnek: {sample[:100]}...")
    print()

# Parçalanmış verimiz üzerinde kontrolleri çalıştır
sanity_check(chunked, "Parçalama Sonrası")

# Bunu her aşamadan sonra çalıştırabilirsiniz:
# sanity_check(cleaned_docs, "Temizleme Sonrası")
# sanity_check(deduped, "Tekilleştirme Sonrası")

## Çalışılmış Örnek: Uçtan Uca Ardışık Düzen
Ham belgelerden (HTML ve kopyalarla) JSONL'ye hazır veriye eksiksiz gözden geçirme.

In [ ]:
print("="*60)
print("EKSİKSİZ VERİ ARDIŞ DÜZEN GÖZDENGEÇİRME")
print("="*60)

# Adım 1: Ham belgelerle başla (dağınık, kopyalar ve HTML ile)
print("\n📥 ADIM 1: Ham Belgeler")
raw_pipeline_docs = [
    {"text": "<p>The cat sat on the mat.</p>", "source": "doc1"},
    {"text": "<p>The cat sat on the mat.</p>", "source": "doc2"},  # tam kopya!
    {"text": "<p>The dog    ran\tin the park.</p>", "source": "doc3"},
    {"text": "The bird flew over the house.", "source": "doc4"}
]
print(f"   Ham belgeler: {len(raw_pipeline_docs)}")
for i, doc in enumerate(raw_pipeline_docs):
    print(f"   {i+1}. {doc['text'][:50]}...")

# Adım 2: Her belgeyi temizle
print("\n🧹 ADIM 2: Temizleme")
for doc in raw_pipeline_docs:
    doc["text"] = clean_text(doc["text"])
print("   HTML kaldırıldı, boşluklar normalize edildi")
for i, doc in enumerate(raw_pipeline_docs):
    print(f"   {i+1}. {doc['text']}")

# Adım 3: Metni çıkar ve tekilleştir
print("\n🔍 ADIM 3: Tekilleştirme")
pipeline_texts = [d["text"] for d in raw_pipeline_docs]
unique_pipeline = dedup_chunks(pipeline_texts)
print(f"   Önce: {len(pipeline_texts)} metin")
print(f"   Sonra:  {len(unique_pipeline)} benzersiz metin")
for i, text in enumerate(unique_pipeline):
    print(f"   {i+1}. {text}")

# Adım 4: Train/val/test'e böl
print("\n📊 ADIM 4: Train/Val/Test Bölmesi")
train_p, val_p, test_p = split_docs(unique_pipeline, train_p=0.34, val_p=0.33, seed=42)
print(f"   Eğitim: {len(train_p)} belge - {train_p}")
print(f"   Doğrulama:   {len(val_p)} belge - {val_p}")
print(f"   Test:  {len(test_p)} belge - {test_p}")

# Adım 5: Parçala (daha uzun belgeler için, burada küçük)
print("\n✂️  ADIM 5: Parçalama")
train_pipeline_chunks = []
for text in train_p:
    chunks = chunk_text(text, max_chars=50, overlap=10)
    train_pipeline_chunks.extend(chunks)
print(f"   Eğitim parçaları: {len(train_pipeline_chunks)}")
for i, chunk in enumerate(train_pipeline_chunks):
    print(f"   Parça {i+1}: {chunk}")

# Adım 6: JSONL kayıtlarını hazırla
print("\n💾 ADIM 6: JSONL Hazırlama")
final_records = [
    {"text": chunk, "split": "train", "length": len(chunk), "source": "example"}
    for chunk in train_pipeline_chunks
]
print(f"   Kaydetmeye hazır: {len(final_records)} kayıt")
print(f"   Örnek kayıt: {final_records[0]}")

print("\n✅ ARDIŞ DÜZEN TAMAMLANDI!")
print(f"   Şununla başlandı: {len(raw_pipeline_docs)} ham belge (kopyalı)")
print(f"   Şununla bitti: {len(final_records)} temiz, tekilleştirilmiş JSONL kaydı")
print(f"   Veri artık Bölüm 8'de tokenizasyon için hazır!")

In [ ]:
def save_jsonl(records, path):
    with open(path, 'w', encoding='utf-8') as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

save_jsonl(chunked, 'toy_corpus.jsonl')
print(len(chunked), 'kayıtla toy_corpus.jsonl yazıldı')

## Hızlı istatistikler

Kopya oranı, en sık kelimeler ve örnek kayıtlar.

In [ ]:
def duplicate_rate(texts):
    hashes = [hash_chunk(t) for t in texts]
    return 1 - (len(set(hashes)) / len(hashes))

def top_words(texts, k=10):
    words = " ".join(texts).lower().split()
    return Counter(words).most_common(k)

texts_all = [r['text'] for r in chunked]
print('Kopya oranı:', duplicate_rate(texts_all))
print('En sık kelimeler:', top_words(texts_all, k=8))
print('Örnek kayıtlar:', chunked[:2])

## Özet

Bu defterde LLM eğitimi için eksiksiz veri hazırlama ardışık düzenini öğrendiniz:

**1. Metin Temizleme:** HTML'i kaldır, boşlukları normalize et, özel karakterleri işle
**2. Tekilleştirme:** Tam kopyaları tanımlamak ve kaldırmak için hash'leme kullan
**3. Train/Val/Test Bölmesi:** Sızıntıyı önlemek için veriyi belge düzeyinde ayır
**4. Örtüşme ile Parçalama:** Uzun metinleri LLM boyutlu parçalara böl, bağlamı koru
**5. Kalite Kontrolleri:** Otomatik akıl sağlığı kontrolleri sorunları erken yakalar
**6. JSONL Formatı:** Veriyi akış dostu bir formatta kaydet

**Ana Kavramlar:**
- **Örtüşme** parça sınırları boyunca bağlamı korur (cümleleri ortadan kesmekten kaçınır)
- **Hash'leme** tekilleştirme için hızlı, deterministik parmak izleri sağlar
- **Belge düzeyinde bölme** ilgili parçaları aynı bölmede bir arada tutar
- **Kalite kontrolleri** HTML sızıntısını, boş parçaları ve boyut anormalliklerini eğitim sorunlarına neden olmadan önce yakalar

**Sonraki Adımlar:**
- Bölüm 8: Tokenizasyon (metni → sayılara dönüştürme)
- Gerçek veri kümelerine ölçeklendirme (Wikipedia, Common Crawl, kitaplar)
- Kullanım durumunuz için farklı örtüşme değerleriyle deneme yapma

Veri ardışık düzeni her harika LLM'nin temelidir!